In [1]:
import os

In [2]:
%pwd

'p:\\OM\\Projects\\End-to-End-ML-\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'p:\\OM\\Projects\\End-to-End-ML-'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataValidationConfig:
    root_dir : Path
    unzip_data_dir : Path
    status_file : str
    all_schema : dict

In [6]:
from src.wine.constants import *
from src.wine.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_file_path = CONFIG_FILE_PATH,
        params_file_path = PARAMS_FILE_PATH,
        schema_file_path = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)

        create_directories([self.config.artifacts_root])

    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation
        schema = self.schema.COLUMNS

        create_directories([config.root_dir])

        data_validation_config = DataValidationConfig(
            root_dir = config.root_dir,
            unzip_data_dir = config.unzip_data_dir,
            status_file = config.status_file,
            all_schema = schema
        )

        return data_validation_config

In [9]:
import pandas as pd
from src.wine.logging import logger

In [13]:
import pandas as pd
from src.wine.logging import logger


class DataValidation:
    def __init__(self, config):
        self.config = config

    def validate_all_columns(self) -> bool:
        try:
            # Read dataset
            data = pd.read_csv(self.config.unzip_data_dir)

            # Actual columns and datatypes from data
            actual_schema = dict(data.dtypes.astype(str))

            # Expected schema from schema.yaml
            expected_schema = self.config.all_schema

            # List to store validation errors
            errors = []

            # Validate columns and datatypes
            for col, dtype in expected_schema.items():

                # Check if column exists
                if col not in actual_schema:
                    errors.append(f"Missing column: {col}")

                # Check datatype
                elif actual_schema[col] != dtype:
                    errors.append(
                        f"{col}: Expected {dtype}, Found {actual_schema[col]}"
                    )

            # Check for extra columns in dataset
            for col in actual_schema.keys():
                if col not in expected_schema:
                    errors.append(f"Unexpected column found: {col}")

            # Validation status
            validation_status = len(errors) == 0  # validation_status is True if no errors and False if any error.

            # Write status and errors to file
            with open(self.config.status_file, "w") as f:
                if validation_status:
                    f.write("Validation status: True")
                else:
                    f.write("Validation status: False\n")
                    for err in errors:
                        f.write(err + "\n")

            logger.info(f"Validation status: {validation_status}")

            if errors:
                logger.info("Validation Errors:")
                for err in errors:
                    logger.info(err)

            return validation_status

        except Exception as e:
            raise e

In [15]:
try : 
    config = ConfigurationManager()     # object creation
    data_validation_config = config.get_data_validation_config()      # stored output of config object's method
    data_validation = DataValidation(config = data_validation_config)     # object creation
    data_validation.validate_all_columns()

except Exception as e : 
    raise e     

[ 2026-06-21 13:52:18,087 : INFO : common : yaml file: config\config.yaml loaded successfully ]
[ 2026-06-21 13:52:18,093 : INFO : common : yaml file: params.yaml loaded successfully ]
[ 2026-06-21 13:52:18,155 : INFO : common : yaml file: schema.yaml loaded successfully ]
[ 2026-06-21 13:52:18,164 : INFO : common : created directory at: artifacts ]
[ 2026-06-21 13:52:18,166 : INFO : common : created directory at: artifacts/data_validation ]
[ 2026-06-21 13:52:18,197 : INFO : 315548686 : Validation status: True ]
